# 🔧 LoRA 파인튜닝

## PEFT와 LoRA

**PEFT** (Parameter-Efficient Fine-Tuning)는 대형 모델을 효율적으로 미세조정하는
**프레임워크**이고, **LoRA**는 그 안의 대표적인 **방법** 중 하나입니다.

```
PEFT (프레임워크)
 ├── LoRA   — 저차원 어댑터 행렬 추가 ← 이 노트북
 ├── OFT    — 직교 변환 행렬 학습     ← 04_osft 노트북
 ├── QLoRA  — 양자화 + LoRA
 └── ...    — Prefix Tuning, Adapters 등
```

## LoRA (Low-Rank Adaptation) 핵심 원리

- 사전 학습된 가중치 행렬 **W**에 저차원 분해 행렬 **ΔW = BA**를 더합니다
- 원래 가중치 W는 고정(freeze)하고, A와 B만 학습합니다
- 전체 파라미터의 **1% 미만**만 학습하면서도 우수한 성능 달성

| 장점 | 설명 |
|------|------|
| **메모리 효율** | 전체 모델 대비 훨씬 적은 GPU 메모리 |
| **빠른 학습** | 학습 파라미터가 적어 수렴이 빠름 |
| **모듈성** | 어댑터만 교체하여 다양한 도메인 적용 |
| **원본 보존** | 기본 모델 가중치를 변경하지 않음 |

### 설정
- **기본 모델**: `Qwen/Qwen3-4B-Instruct-2507`
- **LoRA rank (r)**: 16, **alpha**: 32
- **대상 모듈**: attention (Q/K/V/O) + MLP (gate/up/down)
- **학습 도구**: HuggingFace `TRL SFTTrainer` + `PEFT LoraConfig`

> ⚠️ 이 노트북은 **준비된 데이터 번들**을 사용합니다.


In [ ]:
"""환경 확인 — 00_preflight.ipynb 에서 이미 설치 완료."""

import os
from pathlib import Path

_nb_dir = Path.cwd()
_project_root = _nb_dir
for _p in [_nb_dir] + list(_nb_dir.parents):
    if (_p / "pyproject.toml").exists():
        _project_root = _p
        break

os.environ["RHOAI_PROJECT_ROOT"] = str(_project_root)

try:
    import rhoai_model_training_lab.config as _cfg
    _cfg.PROJECT_ROOT = _project_root
    print(f"✅ 프로젝트 루트: {_project_root}")
except ImportError:
    raise ImportError(
        "❌ 패키지 미설치 — 먼저 00_preflight.ipynb 를 실행하세요."
    )


In [ ]:
"""Load configuration and validate bundle compatibility."""

import os
from pathlib import Path

from rhoai_model_training_lab.config import (
    load_env, load_training_config, load_bundle_config, PROJECT_ROOT,
)
from rhoai_model_training_lab.data import BundleManager

load_env()

# Load LoRA config
lora_config = load_training_config("lora")
model_id = lora_config["model"]["model_id"]
model_revision = lora_config["model"]["model_revision"]

print(f"모델: {model_id} (rev: {model_revision})")
print(f"LoRA r={lora_config['lora']['r']}, alpha={lora_config['lora']['lora_alpha']}")
print(f"대상 모듈: {lora_config['lora']['target_modules']}")
print(f"시드: {lora_config['training']['seed']}")

# Load and validate bundle
release_config = load_bundle_config()
bundle_base = release_config.get("bundle", {}).get("base_path", "data/prepared/tau-knowledge-v1")
bundle_path = PROJECT_ROOT / bundle_base

print(f"\n번들 경로: {bundle_path}")
mgr = BundleManager.load_bundle(bundle_path)
manifest = mgr.manifest

# Compatibility check
compat = mgr.validate_compatibility(model_id)
if compat.errors:
    for err in compat.errors:
        print(f"  ❌ {err}")
    raise RuntimeError(
        "번들 호환성 검증 실패 — 학습을 진행할 수 없습니다.\n"
        "data_preparation/ 노트북에서 올바른 번들을 생성하세요."
    )

print(f"\n✅ 번들 호환성 검증 통과")
print(f"   학습 샘플: {manifest.canonical_train_count}")
print(f"   검증 샘플: {manifest.canonical_validation_count}")
print(f"   번들 버전: {manifest.bundle_version}")

In [ ]:
"""Preview training data with loss mask visualization."""

from transformers import AutoTokenizer

# Load tokenizer for visualization
tokenizer = AutoTokenizer.from_pretrained(model_id, trust_remote_code=True)

# Load backend-specific training data
train_data = mgr.get_training_samples("lora", "train")
val_data = mgr.get_training_samples("lora", "validation")

print(f"LoRA 학습 데이터: {len(train_data)} 샘플")
print(f"LoRA 검증 데이터: {len(val_data)} 샘플")

# Preview first 2 examples with loss mask visualization
print("\n" + "=" * 70)
print("📝 학습 데이터 미리보기 (손실 마스크 시각화)")
print("=" * 70)

for i, sample in enumerate(train_data[:2]):
    messages = sample.get("messages", [])
    print(f"\n--- 샘플 {i+1} ---")
    print(f"메시지 수: {len(messages)}")

    for msg in messages:
        role = msg["role"]
        content = msg.get("content", "") or ""
        # Loss mask: only assistant messages contribute to loss
        is_target = role == "assistant"
        mask_icon = "🎯 [학습 대상]" if is_target else "🚫 [마스킹됨]"

        display_content = content[:200] + "..." if len(content) > 200 else content
        print(f"\n  {mask_icon} [{role}]:")
        print(f"    {display_content}")

        if msg.get("tool_calls"):
            print(f"    🔧 도구 호출: {len(msg['tool_calls'])}건")
            for tc in msg["tool_calls"][:2]:
                fn = tc.get("function", {})
                print(f"       → {fn.get('name', '?')}({fn.get('arguments', '')[:80]})")

    # Token count
    try:
        formatted = tokenizer.apply_chat_template(messages, tokenize=True)
        print(f"\n  토큰 수: {len(formatted)}")
        max_len = lora_config["data"]["max_seq_length"]
        if len(formatted) > max_len:
            print(f"  ⚠️  최대 길이 {max_len} 초과!")
    except Exception:
        pass

print(f"\n{'=' * 70}")
print("손실 마스킹 정책: assistant 응답만 학습 대상")
print("시스템 메시지, 사용자 입력, 도구 관찰값은 손실 계산에서 제외됩니다.")

In [ ]:
"""LoRA 학습 — TRL SFTTrainer + PEFT LoraConfig."""

import time, os
import torch
from pathlib import Path

if not torch.cuda.is_available():
    raise RuntimeError("GPU not detected.")

print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / (1024**3):.1f} GB")

# ── Paths ──
train_file = str(PROJECT_ROOT / lora_config["data"]["train_file"])
val_file = str(PROJECT_ROOT / lora_config["data"]["validation_file"])
output_dir = str(PROJECT_ROOT / lora_config["training_args"]["output_dir"])

# ── Model + Tokenizer ──
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import LoraConfig, get_peft_model, TaskType
from trl import SFTTrainer, SFTConfig
from datasets import load_dataset

print("\n🔧 Loading model...")
tokenizer = AutoTokenizer.from_pretrained(model_id, trust_remote_code=True, padding_side="right")
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    model_id, torch_dtype=torch.bfloat16, device_map="auto",
    trust_remote_code=True, attn_implementation="sdpa",
)
model.config.use_cache = False
print(f"  {model_id} ({model.num_parameters()/1e9:.2f}B params)")

# ── LoRA ──
peft_config = LoraConfig(
    r=lora_config["lora"]["r"],
    lora_alpha=lora_config["lora"]["lora_alpha"],
    lora_dropout=lora_config["lora"]["lora_dropout"],
    target_modules=lora_config["lora"]["target_modules"],
    bias="none", task_type=TaskType.CAUSAL_LM,
)
model = get_peft_model(model, peft_config)
model.print_trainable_parameters()

# ── Dataset ──
dataset = load_dataset("json", data_files={"train": train_file, "validation": val_file})
print(f"\n📊 Train: {len(dataset['train'])} | Val: {len(dataset['validation'])}")

def format_chat(example):
    return {"text": tokenizer.apply_chat_template(
        example["messages"], tokenize=False, add_generation_prompt=False)}

dataset = dataset.map(format_chat, remove_columns=dataset["train"].column_names)

# ── Training ──
ta = lora_config["training_args"]
training_args = SFTConfig(
    output_dir=output_dir,
    num_train_epochs=ta["num_train_epochs"],
    per_device_train_batch_size=ta["per_device_train_batch_size"],
    per_device_eval_batch_size=ta.get("per_device_eval_batch_size", 2),
    gradient_accumulation_steps=ta["gradient_accumulation_steps"],
    learning_rate=ta["learning_rate"],
    weight_decay=ta.get("weight_decay", 0.01),
    warmup_ratio=ta.get("warmup_ratio", 0.03),
    lr_scheduler_type=ta["lr_scheduler_type"],
    max_seq_length=lora_config["data"]["max_seq_length"],
    bf16=ta["bf16"],
    gradient_checkpointing=True,
    logging_steps=ta["logging_steps"],
    eval_strategy="steps",
    eval_steps=ta["eval_steps"],
    save_strategy="steps",
    save_steps=ta["save_steps"],
    save_total_limit=ta["save_total_limit"],
    dataset_text_field="text",
    packing=False,
    report_to="none",
    seed=lora_config["training"]["seed"],
)

trainer = SFTTrainer(
    model=model, args=training_args,
    train_dataset=dataset["train"],
    eval_dataset=dataset["validation"],
    processing_class=tokenizer,
)

print("\n" + "=" * 70)
print("🚀 LoRA Training Start")
print("=" * 70)
print(f"  Epochs: {ta['num_train_epochs']} | Batch: {ta['per_device_train_batch_size']}x{ta['gradient_accumulation_steps']}")
print(f"  LR: {ta['learning_rate']} | LoRA r={lora_config['lora']['r']}, alpha={lora_config['lora']['lora_alpha']}")

start_time = time.time()
training_result = trainer.train()
wall_time = time.time() - start_time

print(f"\n✅ LoRA Training Complete!")
print(f"  Wall time: {wall_time/60:.1f} min | Loss: {training_result.training_loss:.4f}")
print(f"  Peak VRAM: {torch.cuda.max_memory_allocated()/(1024**3):.1f} GB")

trainer.save_model(output_dir)
tokenizer.save_pretrained(output_dir)
print(f"  Adapter saved: {output_dir}")


In [ ]:
"""Training analysis — LoRA loss curves, LR, gradient norm."""

import json as _json
from pathlib import Path

output_dir_path = Path(output_dir)

# trainer.state.log_history contains all logged metrics
log_history = trainer.state.log_history
print(f"Log entries: {len(log_history)}")

train_steps = [e["step"] for e in log_history if "loss" in e]
train_losses = [e["loss"] for e in log_history if "loss" in e]
eval_steps = [e["step"] for e in log_history if "eval_loss" in e]
eval_losses = [e["eval_loss"] for e in log_history if "eval_loss" in e]
lr_steps = [e["step"] for e in log_history if "learning_rate" in e]
lr_values = [e["learning_rate"] for e in log_history if "learning_rate" in e]
grad_steps = [e["step"] for e in log_history if "grad_norm" in e]
grad_norms = [e["grad_norm"] for e in log_history if "grad_norm" in e]

print(f"  Train loss: {len(train_losses)} | Eval: {len(eval_losses)} | LR: {len(lr_values)} | Grad: {len(grad_norms)}")

if train_losses:
    try:
        import matplotlib.pyplot as plt
        import numpy as np

        n_plots = 2 + bool(lr_values) + bool(grad_norms)
        fig, axes = plt.subplots(1, n_plots, figsize=(5 * n_plots, 4.5))
        if n_plots == 1: axes = [axes]
        ax_idx = 0

        # 1. Loss curve
        ax = axes[ax_idx]; ax_idx += 1
        ax.plot(train_steps, train_losses, alpha=0.3, color="#4C72B0", lw=0.8, label="Train (raw)")
        if len(train_losses) > 5:
            sm = [train_losses[0]]
            for v in train_losses[1:]: sm.append(0.1*v + 0.9*sm[-1])
            ax.plot(train_steps, sm, color="#4C72B0", lw=2, label="Train (EMA)")
        if eval_losses:
            ax.plot(eval_steps, eval_losses, color="#DD8452", marker="o", ms=4, lw=2, label="Eval")
        ax.set_xlabel("Step"); ax.set_ylabel("Loss"); ax.set_title("LoRA Loss Curve")
        ax.legend(fontsize=8); ax.grid(True, alpha=0.3)

        # 2. Generalization gap
        ax = axes[ax_idx]; ax_idx += 1
        if eval_losses and len(eval_losses) > 1:
            ax.plot(eval_steps, eval_losses, "o-", color="#DD8452", label="Eval", ms=5)
            it = np.interp(eval_steps, train_steps, train_losses)
            ax.plot(eval_steps, it, "s--", color="#4C72B0", label="Train@eval", ms=4)
            gap = [e-t for e,t in zip(eval_losses, it)]
            ax.fill_between(eval_steps, it, eval_losses, alpha=0.15, color="red",
                           label=f"Gap ({gap[-1]:+.4f})")
            ax.set_title("Generalization Gap")
        else:
            ax.plot(train_steps, train_losses, color="#4C72B0")
            ax.set_title("Train Loss Detail")
        ax.set_xlabel("Step"); ax.set_ylabel("Loss")
        ax.legend(fontsize=8); ax.grid(True, alpha=0.3)

        # 3. LR schedule
        if lr_values:
            ax = axes[ax_idx]; ax_idx += 1
            ax.plot(lr_steps, lr_values, color="#55A868", lw=1.5)
            ax.set_xlabel("Step"); ax.set_ylabel("LR"); ax.set_title("LR Schedule")
            ax.ticklabel_format(axis="y", style="scientific", scilimits=(-4,-4))
            ax.grid(True, alpha=0.3)

        # 4. Gradient norm
        if grad_norms:
            ax = axes[ax_idx]; ax_idx += 1
            ax.plot(grad_steps, grad_norms, color="#C44E52", alpha=0.5, lw=0.8)
            if len(grad_norms) > 5:
                sg = [grad_norms[0]]
                for v in grad_norms[1:]: sg.append(0.1*v + 0.9*sg[-1])
                ax.plot(grad_steps, sg, color="#C44E52", lw=2, label="Smoothed")
                ax.legend(fontsize=8)
            ax.set_xlabel("Step"); ax.set_ylabel("Grad Norm"); ax.set_title("Gradient Norm")
            ax.grid(True, alpha=0.3)

        fig.suptitle("LoRA Training Analysis", fontsize=13, fontweight="bold", y=1.02)
        plt.tight_layout(); plt.show()

    except ImportError:
        for s, l in zip(train_steps[-10:], train_losses[-10:]):
            print(f"  Step {s:>6}: {l:.4f}")

    print("\n" + "=" * 60)
    print("Training Metrics Summary")
    print("=" * 60)
    print(f"  Initial loss:  {train_losses[0]:.4f}")
    print(f"  Final loss:    {train_losses[-1]:.4f}")
    print(f"  Min loss:      {min(train_losses):.4f}")
    reduction = (1 - train_losses[-1] / train_losses[0]) * 100
    print(f"  Reduction:     {reduction:.1f}%")
    if eval_losses:
        print(f"  Final eval:    {eval_losses[-1]:.4f}")
        print(f"  Best eval:     {min(eval_losses):.4f} (step {eval_steps[eval_losses.index(min(eval_losses))]})")
        if len(eval_losses) > 1 and eval_losses[-1] > min(eval_losses) * 1.05:
            print("  ⚠️  Possible overfitting")
    print(f"\n  Total steps:   {train_steps[-1]}")
    print(f"  Peak VRAM:     {torch.cuda.max_memory_allocated()/(1024**3):.1f} GB")
    print(f"  Wall time:     {wall_time/60:.1f} min")
else:
    print("⚠️  No training loss entries found.")


In [ ]:
"""학습 결과 로컬 저장."""

import json as _json

result_summary = {
    "method": "lora",
    "model_id": model_id,
    "bundle_name": manifest.bundle_name,
    "bundle_version": manifest.bundle_version,
    "train_samples": manifest.canonical_train_count,
    "validation_samples": manifest.canonical_validation_count,
    "lora_r": lora_config["lora"]["r"],
    "lora_alpha": lora_config["lora"]["lora_alpha"],
    "learning_rate": lora_config["training_args"]["learning_rate"],
    "num_epochs": lora_config["training_args"]["num_train_epochs"],
    "wall_time_seconds": wall_time,
    "peak_vram_gb": torch.cuda.max_memory_allocated() / (1024**3),
    "gpu_name": torch.cuda.get_device_name(0),
    "final_train_loss": train_losses[-1] if train_losses else None,
    "final_eval_loss": eval_losses[-1] if eval_losses else None,
    "best_eval_loss": min(eval_losses) if eval_losses else None,
}

result_path = Path(output_dir) / "training_result.json"
result_path.parent.mkdir(parents=True, exist_ok=True)
with open(result_path, "w") as f:
    _json.dump(result_summary, f, indent=2, ensure_ascii=False)
print(f"📄 학습 결과 저장: {result_path}")


In [ ]:
"""Verify checkpoint and proceed to export."""

from rich.table import Table
from rich.console import Console

console = Console()

# Verify checkpoint files exist
checkpoint_dir = Path(output_dir)
adapter_config = checkpoint_dir / "adapter_config.json"
adapter_model = checkpoint_dir / "adapter_model.safetensors"

# Find the best/latest checkpoint
checkpoints = sorted(checkpoint_dir.glob("checkpoint-*"), key=lambda p: p.name)
final_adapter = checkpoint_dir

checks = []

# Check adapter config
has_config = adapter_config.exists() or any(
    (cp / "adapter_config.json").exists() for cp in checkpoints
)
checks.append(("어댑터 설정 파일", has_config))

# Check adapter weights
has_weights = adapter_model.exists() or any(
    list((cp).glob("adapter_model*")) for cp in checkpoints
)
checks.append(("어댑터 가중치 파일", has_weights))

# Check tokenizer
has_tokenizer = (checkpoint_dir / "tokenizer_config.json").exists() or any(
    (cp / "tokenizer_config.json").exists() for cp in checkpoints
)
checks.append(("토크나이저 파일", has_tokenizer))

# Verify reload
reload_ok = False
try:
    from peft import PeftModel, PeftConfig

    # Try loading adapter config
    adapter_source = str(checkpoint_dir)
    if not adapter_config.exists() and checkpoints:
        adapter_source = str(checkpoints[-1])

    peft_config = PeftConfig.from_pretrained(adapter_source)
    reload_ok = True
    print(f"\n✅ 어댑터 설정 리로드 성공")
    print(f"   기본 모델: {peft_config.base_model_name_or_path}")
    print(f"   r={peft_config.r}, alpha={peft_config.lora_alpha}")
except Exception as exc:
    print(f"\n⚠️  어댑터 리로드 확인 실패: {exc}")

checks.append(("어댑터 리로드 검증", reload_ok))

# Summary table
table = Table(title="🔍 LoRA 체크포인트 검증", show_header=True)
table.add_column("항목", style="bold")
table.add_column("상태")

for name, ok in checks:
    status = "✅" if ok else "❌"
    table.add_row(name, status)

console.print(table)

all_ok = all(ok for _, ok in checks)
if all_ok:
    print("\n🎉 LoRA 학습이 성공적으로 완료되었습니다!")
    print("\n다음 단계:")
    print("  📓 04_osft_finetuning.ipynb — OFT 학습 (독립적, 동일 데이터)")
    print("  📓 05_grpo_alignment.ipynb — GRPO 추론 정렬 (강화 학습)")
    print("  📓 06_export_and_deploy.ipynb — 모델 내보내기 및 배포")
else:
    print("\n⚠️  일부 검증 항목이 실패했습니다. 위 결과를 확인하세요.")